# 39 · Modelos clínicos: tres escenarios y cinco pliegues

**Revisión 1.0 · Plan metodológico v1.2. Selecciona CPU.**

Se entrenarán regresión logística, XGBoost y MLP en los tres escenarios clínicos del Word: **edad, sexo e IMC**; esos datos más **KL inicial**; y esos datos más **cirugía previa y WOMAC**. Son **45 trabajos**: tres algoritmos × tres escenarios × cinco pliegues, con semilla 2026. La imputación y el escalado se ajustarán dentro del entrenamiento de cada pliegue.

Se necesita la comprobación **38 terminada y guardada en Colab**, incluidas sus tres vistas de aumento. Revisa esas vistas antes de confirmar. La libreta localiza sus registros automáticamente. Un ensayo local o una comprobación sin guardar no satisfacen este requisito.

Los trabajos de la libreta 37 pertenecen al escenario histórico de tres variables y no se reutilizan como resultados de esta campaña ampliada.

Esta libreta queda preparada para una ejecución posterior por el investigador. Las primeras celdas preparan el entorno y comprueban sus entradas. **El entrenamiento solo comienza al escribir la confirmación específica en la celda indicada.**

Se utilizan exclusivamente los datos de desarrollo y sus particiones selladas. Cada trabajo completo se guarda y verifica en la misma carpeta privada de Drive antes de continuar. Una reconexión recupera los trabajos completos compatibles; un trabajo interrumpido se reinicia. Ejecuta una sola sesión por campaña. Los originales se conservan intactos y la prueba reservada permanece cerrada. La calibración, el umbral y el modelo final corresponden a etapas posteriores.

In [ ]:
from pathlib import Path, PurePosixPath
import json, os, re, shutil, stat, subprocess, sys, tempfile, urllib.request, zipfile

CODE_REVISION = '6620536015c6fca88d01dbe68518b74bb6f17df3'
if not re.fullmatch(r'[0-9a-f]{40}', CODE_REVISION) or CODE_REVISION == '0' * 40:
    raise RuntimeError('Esta copia no tiene una revisión publicada. Usa la libreta final entregada.')
if os.environ.get('COLAB_GPU', '0') == '1' or any(os.environ.get(k) for k in ('COLAB_TPU_ADDR', 'TPU_NAME', 'COLAB_TPU_TASK_ID')):
    raise RuntimeError('Selecciona CPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución.')
if not os.environ.get('COLAB_RELEASE_TAG'):
    raise RuntimeError('Esta libreta debe ejecutarse personalmente en Google Colab; no es un ensayo local.')
if 'knee' in sys.modules or any(name.startswith('knee.') for name in sys.modules):
    raise RuntimeError('Reinicia el entorno para cargar la revisión fijada de esta libreta.')
SESSION_ROOT = Path(tempfile.mkdtemp(prefix='knee_methodology39_'))
REPO_DIR = SESSION_ROOT / 'repository'
archive_path = SESSION_ROOT / 'code.zip'
url = 'https://codeload.github.com/AlonsoYess/Knee/zip/' + CODE_REVISION
with urllib.request.urlopen(url, timeout=60) as response, archive_path.open('xb') as target:
    if response.geturl() != url:
        raise RuntimeError('La descarga del código cambió de ubicación.')
    total = 0
    while True:
        chunk = response.read(1024**2)
        if not chunk:
            break
        total += len(chunk)
        if total > 128 * 1024**2:
            raise RuntimeError('El archivo del repositorio supera el límite autorizado.')
        target.write(chunk)
REPO_DIR.mkdir()
with zipfile.ZipFile(archive_path) as archive:
    members = archive.infolist()
    if not members or len(members) > 20000 or sum(i.file_size for i in members) > 512 * 1024**2:
        raise RuntimeError('El contenido del repositorio supera los límites de extracción.')
    seen = set()
    for item in members:
        name = item.filename.rstrip('/')
        parts = name.split('/')
        if not name or '\\' in name or ':' in name or '\0' in name or any(p in ('', '.', '..') for p in parts) or parts[0] != 'Knee-' + CODE_REVISION or name in seen or stat.S_ISLNK(item.external_attr >> 16):
            raise RuntimeError('Se encontró una ruta no permitida en el repositorio.')
        seen.add(name)
        destination = REPO_DIR.joinpath(*parts[1:])
        if item.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        else:
            if len(parts) < 2:
                raise RuntimeError('El archivo del repositorio no tiene una ruta relativa válida.')
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as source, destination.open('xb') as target:
                shutil.copyfileobj(source, target, length=1024**2)
sys.path.insert(0, str(REPO_DIR / 'src'))
from knee import methodology_campaign_colab as orchestration

def checked_run(arguments, **kwargs):
    result = subprocess.run(arguments, capture_output=True, text=True, **kwargs)
    if result.returncode:
        detail = re.sub(r'https?://\S+', '<url>', (result.stdout + result.stderr)[-12000:])
        detail = re.sub(r'(?i)(authorization|token|password|secret)\s*[:=]\s*\S+', r'\1=<redacted>', detail)
        print(detail)
        raise RuntimeError('La comprobación se detuvo. No ejecutes entrenamiento.')
    return result.stdout

print('Revisión exacta del código preparada. No se abrieron datos de participantes.')

CAMPAIGN = 'clinical'


## 1. Preparar el intérprete CPU aislado

Se conservan las dependencias fijadas y se verifica el recurso seleccionado. Esta celda no inicia entrenamiento.

In [ ]:
print('Preparando el intérprete aislado; todavía no se inició entrenamiento…')
runtime = json.loads(checked_run([sys.executable, '-B', str(REPO_DIR / 'scripts/train_methodology_campaign_colab.py'), '--operation', 'setup', '--campaign', CAMPAIGN, '--repository-root', str(REPO_DIR), '--destination', str(SESSION_ROOT / 'isolated_runtime')]))
NEURAL_PYTHON = Path(runtime['python_path'])
RUN_ENV = orchestration.runtime_environment(REPO_DIR, CAMPAIGN)
print('Intérprete preparado. Todavía no se entrenó ningún trabajo.')


## 2. Comprobar la implementación con datos sintéticos

Se prueban los escenarios clínicos, el aumento, la autorización y la recuperación de trabajos. Estas pruebas no ajustan modelos reales. Cualquier fallo u omisión detiene la libreta.

In [ ]:
TEST_ENV = dict(RUN_ENV)
TEST_ENV.update(CUDA_VISIBLE_DEVICES='', COLAB_GPU='0')
CPU_TESTS = json.loads(checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/train_methodology_campaign_colab.py'), '--operation', 'tests', '--campaign', CAMPAIGN, '--repository-root', str(REPO_DIR)], cwd=REPO_DIR, env=TEST_ENV))
print(f"Pruebas sintéticas aprobadas: {CPU_TESTS['tests_run']}; omitidas: {CPU_TESTS['skipped']}.")


## 3. Localizar la comprobación 38 y reconstruir desarrollo

Indica las dos carpetas habituales: registros privados y desarrollo, donde está `transport.json`. Se comprueban los registros completos de la libreta 38 y la integridad del paquete. Si la comprobación 38 no está terminada y guardada, completa primero su última celda.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PERSISTENT_ROOT = orchestration.checked_path(input('Misma carpeta existente de KneeOAI con los registros privados: ').strip())
REMOTE_PACKAGE_DIR = orchestration.checked_path(input('Misma carpeta privada de desarrollo, que contiene transport.json: ').strip())
for selected in (PERSISTENT_ROOT, REMOTE_PACKAGE_DIR):
    if not selected.is_dir() or not selected.is_relative_to(Path('/content/drive')):
        raise RuntimeError('Selecciona carpetas existentes dentro de tu Drive montado.')
proof = json.loads(checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/train_methodology_campaign_colab.py'), '--operation', 'locate', '--campaign', CAMPAIGN, '--repository-root', str(REPO_DIR), '--records-root', str(PERSISTENT_ROOT)], cwd=REPO_DIR, env=RUN_ENV))
PREPARATION_DIR = Path(proof['preparation_dir'])
LOCAL_PACKAGE_DIR = SESSION_ROOT / 'development'
REFERENCE_DIR = SESSION_ROOT / 'reference'
REFERENCE_DIR.mkdir(exist_ok=False)
LOCAL_REFERENCE = REFERENCE_DIR / 'model_inputs_loading_reference_v1.json'
transport = orchestration.stage_development(REMOTE_PACKAGE_DIR, LOCAL_PACKAGE_DIR, LOCAL_REFERENCE)
print('Comprobación 38 localizada. Paquete de desarrollo reconstruido y verificado.')


## 4. Confirmar la campaña y ejecutar sus trabajos

Se entrenarán regresión logística, XGBoost y MLP en los tres escenarios clínicos del Word: **edad, sexo e IMC**; esos datos más **KL inicial**; y esos datos más **cirugía previa y WOMAC**. Son **45 trabajos**: tres algoritmos × tres escenarios × cinco pliegues, con semilla 2026. La imputación y el escalado se ajustarán dentro del entrenamiento de cada pliegue.

**Revisa el alcance y las tres vistas del aumento de la libreta 38 antes de confirmar.** La confirmación cubre solo esta campaña. Se mostrará el progreso real por trabajo y una señal periódica de actividad. Cada trabajo completo se copia a Drive y se verifica antes del siguiente. Si hay una interrupción, conserva el mensaje y vuelve a ejecutar esta misma revisión con las mismas carpetas.

In [ ]:
confirmation = input('Escribe ' + orchestration.CONFIRMATIONS[CAMPAIGN] + ' para ejecutar solo esta campaña: ').strip()
if confirmation != orchestration.CONFIRMATIONS[CAMPAIGN]:
    raise RuntimeError('No se confirmó la campaña. No se inició entrenamiento.')
import queue, threading, time, uuid

def training_run(arguments, *, cwd, env):
    # Forward actual stderr progress; stdout contains the final structured record.
    process = subprocess.Popen(arguments, cwd=cwd, env=env, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    messages = queue.Queue()
    def forward(stream, channel):
        try:
            for line in iter(stream.readline, ''):
                messages.put((channel, line))
        finally:
            stream.close()
            messages.put((channel, None))
    for stream, channel in ((process.stdout, 'stdout'), (process.stderr, 'stderr')):
        threading.Thread(target=forward, args=(stream, channel), daemon=True).start()
    started, last_heartbeat, closed = time.monotonic(), time.monotonic(), set()
    stdout_lines = []
    try:
        while len(closed) < 2:
            try:
                channel, line = messages.get(timeout=1)
            except queue.Empty:
                channel, line = None, None
            if channel is not None:
                if line is None:
                    closed.add(channel)
                elif channel == 'stdout':
                    stdout_lines.append(line)
                else:
                    safe = re.sub(r'https?://\S+', '<url>', line.rstrip())
                    safe = re.sub(r'(?i)(authorization|token|password|secret)\s*[:=]\s*\S+', r'\1=<redacted>', safe)
                    print(safe, flush=True)
            now = time.monotonic()
            if now - last_heartbeat >= 30 and process.poll() is None:
                print(f'Subproceso activo; tiempo transcurrido: {int(now - started)} s. No es un porcentaje de avance.', flush=True)
                last_heartbeat = now
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    if status:
        raise RuntimeError('El entrenamiento se detuvo. Conserva el error y los trabajos completos guardados en Drive; no cambies la receta.')
    return ''.join(stdout_lines)

LOCAL_OUTPUT = SESSION_ROOT / ('methodology_campaign_' + uuid.uuid4().hex)
TEST_RECEIPT = SESSION_ROOT / 'synthetic_test_receipt.json'
RUNTIME_RECEIPT = SESSION_ROOT / 'setup_receipt.json'
orchestration._save(TEST_RECEIPT, CPU_TESTS)
orchestration._save(RUNTIME_RECEIPT, runtime)
arguments = [str(NEURAL_PYTHON), '-u', '-B', str(REPO_DIR / 'scripts/train_methodology_campaign_colab.py'), '--operation', 'train', '--campaign', CAMPAIGN, '--repository-root', str(REPO_DIR), '--preparation-dir', str(PREPARATION_DIR), '--package-dir', str(LOCAL_PACKAGE_DIR), '--reference-path', str(LOCAL_REFERENCE), '--output-dir', str(LOCAL_OUTPUT), '--records-root', str(PERSISTENT_ROOT), '--code-revision', CODE_REVISION, '--test-receipt', str(TEST_RECEIPT), '--runtime-receipt', str(RUNTIME_RECEIPT), '--approval-statement', orchestration.APPROVAL_STATEMENTS[CAMPAIGN]]
if CAMPAIGN == 'densenet_frozen':
    WEIGHTS_ROOT = SESSION_ROOT / 'verified_public_weights'
    arguments.extend(['--weights-root', str(WEIGHTS_ROOT)])
print('Iniciando únicamente la campaña confirmada del plan v1.2. Cada trabajo completo se guardará en Drive…', flush=True)
training_log = training_run(arguments, cwd=REPO_DIR, env=RUN_ENV)
FINAL_CHECK = json.loads((LOCAL_OUTPUT / orchestration.SUMMARY_FILENAME).read_text(encoding='utf-8'))
orchestration.validate_result(FINAL_CHECK)
print('Campaña terminada. Resultados privados guardados y verificados en Drive.')


## 5. Mostrar el resultado

La selección utiliza la media no ponderada de AP de los cinco pliegues de cada candidato. La AP al reunir las predicciones fuera de pliegue se informa de forma descriptiva. Son resultados de desarrollo; la evaluación final se realizará posteriormente.

Comparte únicamente el **JSON final**. Las predicciones individuales y los modelos permanecen privados.

In [ ]:
print(json.dumps(FINAL_CHECK, ensure_ascii=False, indent=2))
print('Resultados de desarrollo guardados. No se abrió la prueba reservada ni se ajustó la calibración final.')
